In [1]:
TRAINING_DATA_PATH = f"G:\Reasrch_Paper_2026\dataset\Training"
TESTING_DATA_PATH = f"G:\Reasrch_Paper_2026\dataset\Testing"

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
from torchvision import models
import torch.optim as optim


In [3]:
!pip install imutils

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for imutils: filename=imutils-0.5.4-py3-none-any.whl size=25891 sha256=fc53142b0efc5ddba4fe841046931aea8b2c90520141e8c716cbb290b5cab56e
  Stored in directory: c:\users\exposotic\appdata\local\pip\cache\wheels\4b\a5\2d\4a070a801d3a3d93f033d3ee9728f470f514826e89952df3ea
Successfully built imutils


In [4]:
!pip install opencv-python-headless imutils

   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.1 MB ? eta -:--:--
    --------------------------------------- 0.8/40.1 MB 2.2 MB/s eta 0:00:18
   - -------------------------------------- 1.3/40.1 MB 2.7 MB/s eta 0:00:15
   - -------------------------------------- 1.8/40.1 MB 2.5 MB/s eta 0:00:16
   -- ------------------------------------- 2.4/40.1 MB 2.6 MB/s eta 0:00:15
   -- ------------------------------------- 2.9/40.1 MB 2.6 MB/s eta 0:00:15
   --- ------------------------------------ 3.4/40.1 MB 2.5 MB/s eta 0:00:15
   ---- ----------------------------------- 4.2/40.1 MB 2.7 MB/s eta 0:00:14
   ---- ----------------------------------- 4.7/40.1 MB 2.7 MB/s eta 0:00:14
   ----- ---------------------------------- 5.2/40.1 MB 2.7 MB/s eta 0:00:13
   ------ --------------------------------- 6.0/40.1 MB 2.8 MB/s eta 0:00:13
   ------ --------------------------------- 6.6/40.1 MB 2.8 MB/s eta 0:00:12
   ------- --

In [5]:
import cv2
import numpy as np
import imutils
from PIL import Image

class CropBrainContour(object):
    def __call__(self, img):
        # Convert PIL image to CV2 format (numpy)
        img_cv2 = np.array(img)
        img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_RGB2BGR)

        gray = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
        thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=2)
        cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)

        if len(cnts) > 0:
            c = max(cnts, key=cv2.contourArea)
            extLeft  = tuple(c[c[:, :, 0].argmin()][0])
            extRight = tuple(c[c[:, :, 0].argmax()][0])
            extTop   = tuple(c[c[:, :, 1].argmin()][0])
            extBot   = tuple(c[c[:, :, 1].argmax()][0])
            img_cv2 = img_cv2[extTop[1]:extBot[1], extLeft[0]:extRight[0]]

        # Convert back to PIL for the rest of the transforms
        return Image.fromarray(cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB))

In [6]:
# Hyperparameters
img_size = 224
batch_size = 32
test_split_ratio = 0.2
random_seed = 42

In [10]:
# Define transforms
data_transforms = transforms.Compose([
    CropBrainContour(),                  # call crop class
    transforms.Resize((224, 224)),       # Resize to standard CNN input size
    transforms.ToTensor(),                # Convert image to PyTorch tensor (0-1)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

# Training Transforms (Add variation to help the model generalize)
train_transforms = transforms.Compose([
    CropBrainContour(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),      # Flip images randomly
    transforms.RandomRotation(15),               # Slight rotations (common in medical scans)
    transforms.ColorJitter(brightness=0.1, contrast=0.1), # Small lighting variations
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test Transforms (Keep strictly original for testing)
val_transforms = transforms.Compose([
    CropBrainContour(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [11]:
import random
from copy import deepcopy

# 1. Load the dataset twice with DIFFERENT transforms
train_full_dataset = datasets.ImageFolder(root=TRAINING_DATA_PATH, transform=train_transforms)
val_full_dataset = datasets.ImageFolder(root=TRAINING_DATA_PATH, transform=val_transforms)

# 2. Define the split ratios
train_size = int(0.8 * len(train_full_dataset))
val_size = len(train_full_dataset) - train_size

# 3. Split the indices (reproducibly)
torch.manual_seed(random_seed)
# We split the datasets. Since the indices are the same, they point to the same images
train_dataset, _ = random_split(train_full_dataset, [train_size, val_size])
_, val_dataset = random_split(val_full_dataset, [train_size, val_size])

# 4. Handle your testing set separately
Testing_full_dataset = datasets.ImageFolder(root=TESTING_DATA_PATH, transform=val_transforms)

print(f"Classes found: {train_full_dataset.classes}")
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")


Classes found: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training set size: 4480
Validation set size: 1120


In [12]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(Testing_full_dataset, batch_size=batch_size, shuffle=False)

# Optional: Check a batch shape from the train_loader
# for images, labels in train_loader:
#     print(f"Batch shape: {images.shape}")
#     print(f"the label ={labels.shape}")
#     break

In [13]:
# To use windows GPU faster than typical google colab CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# Load the VGG16 model with pretrained weights
VGG_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

num_classes = 4

# Freeze all parameters in the feature extractor
for param in VGG_model.features.parameters():
    param.requires_grad = False

# Replace the final layer of the classifier
in_features = VGG_model.classifier[6].in_features
VGG_model.classifier[6] = nn.Linear(in_features, num_classes)

# Loss Function and Optimizer Configuration
optimizer = optim.SGD(VGG_model.classifier.parameters(), lr=0.001, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# Send the model to device
VGG_model = VGG_model.to(device)

num_epochs = 30

# Early Stopping Parameters
patience = 5
min_delta = 0.001 # Minimum change in the monitored quantity to qualify as an improvement.
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    VGG_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = VGG_model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100 * correct / total

    # Validation phase
    VGG_model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = VGG_model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = 100 * val_correct / val_total

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

    # Early stopping check
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to no improvement in validation loss.')
            break


Epoch [1/30], Train Loss: 0.6621, Train Acc: 73.71%, Val Loss: 0.3912, Val Acc: 85.27%
Epoch [2/30], Train Loss: 0.4299, Train Acc: 83.91%, Val Loss: 0.3199, Val Acc: 88.04%
Epoch [3/30], Train Loss: 0.3623, Train Acc: 86.23%, Val Loss: 0.2781, Val Acc: 90.71%
Epoch [4/30], Train Loss: 0.3213, Train Acc: 88.17%, Val Loss: 0.2445, Val Acc: 91.43%
Epoch [5/30], Train Loss: 0.2813, Train Acc: 89.33%, Val Loss: 0.2475, Val Acc: 89.91%
Epoch [6/30], Train Loss: 0.2529, Train Acc: 90.54%, Val Loss: 0.2078, Val Acc: 92.77%
Epoch [7/30], Train Loss: 0.2380, Train Acc: 91.32%, Val Loss: 0.1779, Val Acc: 93.93%
Epoch [8/30], Train Loss: 0.2220, Train Acc: 91.90%, Val Loss: 0.1663, Val Acc: 94.55%
Epoch [9/30], Train Loss: 0.2000, Train Acc: 92.41%, Val Loss: 0.1494, Val Acc: 94.38%
Epoch [10/30], Train Loss: 0.1962, Train Acc: 92.63%, Val Loss: 0.1509, Val Acc: 94.73%
Epoch [11/30], Train Loss: 0.1713, Train Acc: 93.88%, Val Loss: 0.1278, Val Acc: 95.98%
Epoch [12/30], Train Loss: 0.1567, Train 

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

VGG_model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = VGG_model(inputs)

        # required for AUC-ROC
        probs = F.softmax(outputs, dim=1)

        _, predicted = torch.max(outputs.data, 1)

        # convery to numpy for sklearn
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# macro average to treat each class equal
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

test_accuracy = accuracy_score(all_labels, all_preds) * 100

print(f'Test Accuracy:  {test_accuracy:.2f}%')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'AUC-ROC:   {auc_roc:.4f}')

print("\n Classification Report")
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
print(classification_report(all_labels, all_preds, target_names=class_names))

Test Accuracy:  92.00%
Precision: 0.9234
Recall:    0.9200
F1-Score:  0.9179
AUC-ROC:   0.9858

 Classification Report
              precision    recall  f1-score   support

      glioma       0.96      0.76      0.85       400
  meningioma       0.85      0.93      0.89       400
     notumor       0.92      1.00      0.96       400
   pituitary       0.96      0.99      0.98       400

    accuracy                           0.92      1600
   macro avg       0.92      0.92      0.92      1600
weighted avg       0.92      0.92      0.92      1600



In [ ]:
# 1. Load the DenseNet121 model with pretrained weights
DenseNet_model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

num_classes = 4

# 2. Freeze all parameters in the feature extractor (DenseNet's "features" block)
for param in DenseNet_model.features.parameters():
    param.requires_grad = False

# 3. Replace the final layer of the classifier
in_features = DenseNet_model.classifier.in_features
DenseNet_model.classifier = nn.Linear(in_features, num_classes)

# 4. Loss Function and Optimizer Configuration
optimizer = optim.Adam(DenseNet_model.classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 5. Send the model to device
DenseNet_model = DenseNet_model.to(device)

num_epochs = 30

# 6. Early Stopping Parameters
patience = 5
min_delta = 0.001
best_val_loss = float('inf')
epochs_no_improve = 0

# --- Training Loop ---
for epoch in range(num_epochs):
    DenseNet_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = DenseNet_model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100 * correct / total

    # Validation phase
    DenseNet_model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = DenseNet_model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = 100 * val_correct / val_total

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

    # Early stopping check
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to no improvement in validation loss.')
            break

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 34.0MB/s]


Epoch [1/30], Train Loss: 0.7946, Train Acc: 71.16%, Val Loss: 0.5573, Val Acc: 83.57%
Epoch [2/30], Train Loss: 0.5203, Train Acc: 82.21%, Val Loss: 0.4573, Val Acc: 85.54%
Epoch [3/30], Train Loss: 0.4451, Train Acc: 84.93%, Val Loss: 0.4824, Val Acc: 83.75%
Epoch [4/30], Train Loss: 0.3995, Train Acc: 85.65%, Val Loss: 0.3809, Val Acc: 87.32%
Epoch [5/30], Train Loss: 0.3769, Train Acc: 86.76%, Val Loss: 0.3580, Val Acc: 87.59%
Epoch [6/30], Train Loss: 0.3759, Train Acc: 86.21%, Val Loss: 0.3732, Val Acc: 86.79%
Epoch [7/30], Train Loss: 0.3530, Train Acc: 87.12%, Val Loss: 0.3168, Val Acc: 89.64%
Epoch [8/30], Train Loss: 0.3354, Train Acc: 88.35%, Val Loss: 0.3162, Val Acc: 89.46%
Epoch [9/30], Train Loss: 0.3351, Train Acc: 87.79%, Val Loss: 0.3014, Val Acc: 89.29%
Epoch [10/30], Train Loss: 0.3248, Train Acc: 88.24%, Val Loss: 0.3192, Val Acc: 88.39%
Epoch [11/30], Train Loss: 0.3241, Train Acc: 87.95%, Val Loss: 0.3171, Val Acc: 89.02%
Epoch [12/30], Train Loss: 0.3125, Train 

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

DenseNet_model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = DenseNet_model(inputs)

        # required for AUC-ROC
        probs = F.softmax(outputs, dim=1)

        _, predicted = torch.max(outputs.data, 1)

        # convery to numpy for sklearn
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# macro average to treat each class equal
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

test_accuracy = accuracy_score(all_labels, all_preds) * 100

print(f'Test Accuracy:  {test_accuracy:.2f}%')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'AUC-ROC:   {auc_roc:.4f}')

print("\n Classification Report")
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
print(classification_report(all_labels, all_preds, target_names=class_names))

Test Accuracy:  82.94%
Precision: 0.8497
Recall:    0.8294
F1-Score:  0.8258
AUC-ROC:   0.9585

 Classification Report
              precision    recall  f1-score   support

      glioma       0.96      0.61      0.74       400
  meningioma       0.70      0.83      0.76       400
     notumor       0.80      0.99      0.89       400
   pituitary       0.93      0.89      0.91       400

    accuracy                           0.83      1600
   macro avg       0.85      0.83      0.83      1600
weighted avg       0.85      0.83      0.83      1600



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Load the EfficientNetV2-S model with pretrained weights
effnet_model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

num_classes = 4

# 2. Freeze all parameters in the feature extractor
for param in effnet_model.parameters():
    param.requires_grad = False

# 3. Replace the final classifier head
in_features = effnet_model.classifier[1].in_features
effnet_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, num_classes)
)

# 4. Loss Function and Optimizer Configuration
optimizer = optim.SGD(effnet_model.classifier.parameters(), lr=0.001, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# 5. Send the model to device
effnet_model = effnet_model.to(device)

num_epochs = 30

# 6. Early Stopping Parameters
patience = 5
min_delta = 0.001
best_val_loss = float('inf')
epochs_no_improve = 0

# Training Loop
for epoch in range(num_epochs):
    effnet_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = effnet_model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100 * correct / total

    # Validation phase
    effnet_model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = effnet_model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = 100 * val_correct / val_total

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

    # Early stopping check
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to no improvement in validation loss.')
            break

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 177MB/s]


Epoch [1/30], Train Loss: 1.1362, Train Acc: 59.29%, Val Loss: 0.9541, Val Acc: 73.93%
Epoch [2/30], Train Loss: 0.8411, Train Acc: 75.42%, Val Loss: 0.7939, Val Acc: 76.70%
Epoch [3/30], Train Loss: 0.7335, Train Acc: 76.90%, Val Loss: 0.7246, Val Acc: 77.50%
Epoch [4/30], Train Loss: 0.6757, Train Acc: 77.83%, Val Loss: 0.6935, Val Acc: 77.41%
Epoch [5/30], Train Loss: 0.6494, Train Acc: 77.54%, Val Loss: 0.6564, Val Acc: 78.12%
Epoch [6/30], Train Loss: 0.6204, Train Acc: 78.24%, Val Loss: 0.6247, Val Acc: 79.11%
Epoch [7/30], Train Loss: 0.6022, Train Acc: 79.62%, Val Loss: 0.6275, Val Acc: 78.39%
Epoch [8/30], Train Loss: 0.5800, Train Acc: 80.29%, Val Loss: 0.6062, Val Acc: 79.55%
Epoch [9/30], Train Loss: 0.5743, Train Acc: 79.89%, Val Loss: 0.5798, Val Acc: 81.07%
Epoch [10/30], Train Loss: 0.5656, Train Acc: 79.78%, Val Loss: 0.5763, Val Acc: 80.45%
Epoch [11/30], Train Loss: 0.5541, Train Acc: 79.98%, Val Loss: 0.5804, Val Acc: 80.36%
Epoch [12/30], Train Loss: 0.5536, Train 

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

effnet_model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass through EfficientNetV2-S
        outputs = effnet_model(inputs)

        # Required for AUC-ROC (probabilities per class)
        probs = F.softmax(outputs, dim=1)

        # Get the predicted class index
        _, predicted = torch.max(outputs.data, 1)

        # Convert to numpy for sklearn compatibility
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Calculate metrics
# macro average treats each class (glioma, meningioma, etc.) with equal weight
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

# AUC-ROC using One-vs-Rest (OvR) strategy for multiclass
auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

test_accuracy = accuracy_score(all_labels, all_preds) * 100

# Results output
print(f'Test Accuracy:  {test_accuracy:.2f}%')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'AUC-ROC:   {auc_roc:.4f}')

print("\nClassification Report")
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
print(classification_report(all_labels, all_preds, target_names=class_names))

Test Accuracy:  77.50%
Precision: 0.7810
Recall:    0.7750
F1-Score:  0.7668
AUC-ROC:   0.9354

Classification Report
              precision    recall  f1-score   support

      glioma       0.84      0.60      0.70       400
  meningioma       0.70      0.61      0.65       400
     notumor       0.73      0.99      0.84       400
   pituitary       0.85      0.90      0.88       400

    accuracy                           0.78      1600
   macro avg       0.78      0.77      0.77      1600
weighted avg       0.78      0.78      0.77      1600



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        # Layers designed for 224x224 input
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(512 * 7 * 7, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))
        x = x.view(-1, 512 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CustomCNN_model = CustomCNN(num_classes=4).to(device)


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CustomCNN_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 3. EARLY STOPPING PARAMETERS
num_epochs = 35
patience = 5
min_delta = 0.001
best_val_loss = float('inf')
epochs_no_improve = 0


for epoch in range(num_epochs):
    # Training Phase
    CustomCNN_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = CustomCNN_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100 * correct / total

    # Validation Phase
    CustomCNN_model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = CustomCNN_model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = 100 * val_correct / val_total

    # Update Learning Rate Scheduler
    scheduler.step(epoch_val_loss)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Acc: {epoch_train_acc:.2f}%, Val Acc: {epoch_val_acc:.2f}%, Val Loss: {epoch_val_loss:.4f}')

     # Early stopping check
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to no improvement in validation loss.')
            break


Epoch [1/35], Train Acc: 52.54%, Val Acc: 69.29%, Val Loss: 0.7782
Epoch [2/35], Train Acc: 66.88%, Val Acc: 74.91%, Val Loss: 0.6949
Epoch [3/35], Train Acc: 72.05%, Val Acc: 76.79%, Val Loss: 0.5962
Epoch [4/35], Train Acc: 75.27%, Val Acc: 79.20%, Val Loss: 0.5106
Epoch [5/35], Train Acc: 78.28%, Val Acc: 74.20%, Val Loss: 0.6816
Epoch [6/35], Train Acc: 79.26%, Val Acc: 85.80%, Val Loss: 0.3986
Epoch [7/35], Train Acc: 82.48%, Val Acc: 73.04%, Val Loss: 0.7719
Epoch [8/35], Train Acc: 82.32%, Val Acc: 54.29%, Val Loss: 1.6881
Epoch [9/35], Train Acc: 83.28%, Val Acc: 83.39%, Val Loss: 0.4114
Epoch [10/35], Train Acc: 87.37%, Val Acc: 91.79%, Val Loss: 0.2378
Epoch [11/35], Train Acc: 88.62%, Val Acc: 90.80%, Val Loss: 0.2413
Epoch [12/35], Train Acc: 89.00%, Val Acc: 92.77%, Val Loss: 0.2072
Epoch [13/35], Train Acc: 89.75%, Val Acc: 90.80%, Val Loss: 0.2703
Epoch [14/35], Train Acc: 90.20%, Val Acc: 88.30%, Val Loss: 0.3398
Epoch [15/35], Train Acc: 90.04%, Val Acc: 91.61%, Val Lo

In [16]:
# Testing phase
CustomCNN_model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = CustomCNN_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f'\nFinal Custom CNN Test Accuracy: {100 * test_correct / test_total:.2f}%')

NameError: name 'CustomCNN_model' is not defined

In [ ]:
torch.save(CustomCNN_model.state_dict(), 'custom_cnn.pth')
print("Model saved successfully to 'custom_cnn.pth'!")

Model saved successfully to 'custom_cnn.pth'!


#Tri-Fusion Learning:


###1) Feature-Level Concatenation Fusion

In [14]:
import torch
import torch.nn as nn
from torchvision import models

class TriFusionEfficient(nn.Module):
    def __init__(self, num_classes=4):
        super(TriFusionEfficient, self).__init__()

        # 1. Load Pre-trained Backbones
        self.vgg = models.vgg16(weights='IMAGENET1K_V1')
        self.effnet = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
        self.densenet = models.densenet121(weights='IMAGENET1K_V1')

        # 2. Extract Features by Replacing Classification Heads
        # VGG16: Output is 4096 (from the last FC layer)
        self.vgg.classifier[6] = nn.Identity()

        # EfficientNetV2-S: Output is 1280 (from the classifier's input)
        self.effnet.classifier = nn.Identity()

        # DenseNet121: Output is 1024 (from the classifier's input)
        self.densenet.classifier = nn.Identity()

        # 3. Unified Classifier Head
        # Concatenated size: 4096 (VGG) + 1280 (EffNet) + 1024 (DenseNet) = 6400
        self.fusion_classifier = nn.Sequential(
            nn.Linear(6400, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes) # 4 Classes for MRI
        )

    def forward(self, x):
        # Forward pass through each backbone
        # We use .clone() or separate variables to ensure no gradient interference
        f_vgg = self.vgg(x)
        f_eff = self.effnet(x)
        f_dense = self.densenet(x)

        # Concatenate features along the feature dimension (dim=1)
        combined = torch.cat((f_vgg, f_eff, f_dense), dim=1)

        # Final classification
        out = self.fusion_classifier(combined)
        return out

num_classes = 4
model = TriFusionEfficient(num_classes=num_classes)
print(f"Concatenated Feature Size: 6400")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

0.0%

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\Exposotic/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\Exposotic/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth


100.0%
1.2%

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Exposotic/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100.0%


Concatenated Feature Size: 6400
Total Parameters: 168,473,364


In [15]:
model = TriFusionEfficient(num_classes=4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Differential Learning Rates
optimizer = optim.Adam([
    {'params': model.vgg.parameters(), 'lr': 1e-5},
    {'params': model.effnet.parameters(), 'lr': 1e-5},
    {'params': model.densenet.parameters(), 'lr': 1e-5},
    {'params': model.fusion_classifier.parameters(), 'lr': 1e-4}
])

# 3. TRAINING PARAMETERS & EARLY STOPPING
num_epochs = 30
patience = 5
min_delta = 0.001
best_val_loss = float('inf')
epochs_no_improve = 0

print("Staring Training")

# 4. TRAINING LOOP
for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    epoch_val_loss = val_running_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val

    # 5. LOGGING RESULTS
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
    print(f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')
    print("-" * 30)

    # 6. EARLY STOPPING CHECK
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        # Save best weights for Overleaf/Research Paper results
        torch.save(model.state_dict(), 'best_tri_fusion_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to lack of improvement.')
            break

Staring Training
Epoch [1/30]
Train Loss: 0.5545, Train Acc: 79.06%
Val Loss: 0.3590, Val Acc: 85.62%
------------------------------
Epoch [2/30]
Train Loss: 0.2497, Train Acc: 90.83%
Val Loss: 0.1313, Val Acc: 95.98%
------------------------------
Epoch [3/30]
Train Loss: 0.1584, Train Acc: 94.26%
Val Loss: 0.0782, Val Acc: 97.23%
------------------------------
Epoch [4/30]
Train Loss: 0.0942, Train Acc: 96.41%
Val Loss: 0.0710, Val Acc: 97.59%
------------------------------
Epoch [5/30]
Train Loss: 0.0708, Train Acc: 97.34%
Val Loss: 0.0420, Val Acc: 99.02%
------------------------------
Epoch [6/30]
Train Loss: 0.0591, Train Acc: 97.77%
Val Loss: 0.0288, Val Acc: 99.11%
------------------------------
Epoch [7/30]
Train Loss: 0.0421, Train Acc: 98.55%
Val Loss: 0.0401, Val Acc: 98.84%
------------------------------
Epoch [8/30]
Train Loss: 0.0239, Train Acc: 99.40%
Val Loss: 0.0232, Val Acc: 99.46%
------------------------------
Epoch [9/30]
Train Loss: 0.0205, Train Acc: 99.29%
Val 

In [ ]:
# This is the code for saving the model into 

import os

# Assume 'model' is your trained PyTorch CNN model
# and 'EPOCH' is the current epoch number.

# Define the save directory and filename
save_dir = r"G:\Reasrch_Paper_2026" # Use 'r' for raw string to handle backslashes
file_name = f"best_tri_fusion_model1.pth"
save_path = os.path.join(save_dir, file_name)

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

# Save only the model's state_dict
torch.save(model.state_dict(), save_path)

In [18]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

# 1. Set model to evaluation mode
model.eval()

all_labels = []
all_preds = []
all_probs = []

# 2. Evaluation Loop
with torch.no_grad():
    for inputs, labels in test_loader: # Ensure you use test_loader or val_loader
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass through the Tri-Fusion model
        outputs = model(inputs)

        # Get probabilities for AUC-ROC calculation
        probs = F.softmax(outputs, dim=1)

        # Get the predicted class index
        _, predicted = torch.max(outputs.data, 1)

        # Convert to numpy and extend lists
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# 3. Calculate Comprehensive Metrics
# macro average is ideal for your balanced 4-class MRI dataset
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

# Multi-class AUC-ROC (OvR strategy)
auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

test_accuracy = accuracy_score(all_labels, all_preds) * 100

# 4. Results Output for Research Paper
print(f'--- Tri-Fusion Evaluation Results ---')
print(f'Test Accuracy:  {test_accuracy:.2f}%')
print(f'Precision:      {precision:.4f}')
print(f'Recall:         {recall:.4f}')
print(f'F1-Score:       {f1:.4f}')
print(f'AUC-ROC:        {auc_roc:.4f}')

print("\nClassification Report")
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
print(classification_report(all_labels, all_preds, target_names=class_names))

--- Tri-Fusion Evaluation Results ---
Test Accuracy:  94.44%
Precision:      0.9486
Recall:         0.9444
F1-Score:       0.9431
AUC-ROC:        0.9930

Classification Report
              precision    recall  f1-score   support

      glioma       0.99      0.81      0.89       400
  meningioma       0.90      0.98      0.94       400
     notumor       0.91      1.00      0.95       400
   pituitary       0.99      0.99      0.99       400

    accuracy                           0.94      1600
   macro avg       0.95      0.94      0.94      1600
weighted avg       0.95      0.94      0.94      1600



###2) Attention Weighted Fusion

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionTriFusion(nn.Module):
    def __init__(self, num_classes=4):
        super(AttentionTriFusion, self).__init__()

        # 1. Backbones (Pre-trained)
        self.vgg = models.vgg16(weights='IMAGENET1K_V1')
        self.effnet = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
        self.densenet = models.densenet121(weights='IMAGENET1K_V1')

        self.vgg.classifier[6] = nn.Identity()
        self.effnet.classifier = nn.Identity()
        self.densenet.classifier = nn.Identity()

        # 2. Projection Layers (Standardizing dimensions to 512)
        self.proj_vgg = nn.Linear(4096, 512)
        self.proj_eff = nn.Linear(1280, 512)
        self.proj_den = nn.Linear(1024, 512)

        # 3. Attention Gate
        # Now it looks at the 1536 combined projected features
        self.attention_gate = nn.Sequential(
            nn.Linear(512 * 3, 256),
            nn.ReLU(),
            nn.Linear(256, 3),
            nn.Softmax(dim=1)
        )

        # 4. Final Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Extract features
        vgg_feat = self.vgg(x)
        eff_feat = self.effnet(x)
        den_feat = self.densenet(x)

        # Project to common dimension
        p_vgg = F.relu(self.proj_vgg(vgg_feat))
        p_eff = F.relu(self.proj_eff(eff_feat))
        p_den = F.relu(self.proj_den(den_feat))

        # Concatenate for the gate
        combined_proj = torch.cat((p_vgg, p_eff, p_den), dim=1)

        # Calculate attention weights
        attn_weights = self.attention_gate(combined_proj)

        # Apply weights to the projected features
        v_weighted = p_vgg * attn_weights[:, 0:1]
        e_weighted = p_eff * attn_weights[:, 1:2]
        d_weighted = p_den * attn_weights[:, 2:3]

        # Re-concatenate weighted features
        final_features = torch.cat((v_weighted, e_weighted, d_weighted), dim=1)

        return self.classifier(final_features)

In [22]:
model = AttentionTriFusion(num_classes=4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.vgg.parameters(), 'lr': 1e-5},
    {'params': model.effnet.parameters(), 'lr': 1e-5},
    {'params': model.densenet.parameters(), 'lr': 1e-5},
    {'params': model.attention_gate.parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
])

num_epochs = 30
patience = 5
min_delta = 0.001
best_val_loss = float('inf')
epochs_no_improve = 0

print("Starting Training")

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # Track training metrics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    # Calculate Epoch-level Training Metrics
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    # Calculate Epoch-level Validation Metrics
    epoch_val_loss = val_running_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val

    # --- 5. LOGGING RESULTS (Formatted to match your original style) ---
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
    print(f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')
    print("-" * 30)

    # --- 6. EARLY STOPPING CHECK ---
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_attention_fusion.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs due to lack of improvement.')
            break

Starting Training
Epoch [1/30]
Train Loss: 0.8317, Train Acc: 67.68%
Val Loss: 0.3848, Val Acc: 87.95%
------------------------------
Epoch [2/30]
Train Loss: 0.3376, Train Acc: 88.82%
Val Loss: 0.2171, Val Acc: 93.30%
------------------------------
Epoch [3/30]
Train Loss: 0.2245, Train Acc: 91.99%
Val Loss: 0.1567, Val Acc: 95.00%
------------------------------
Epoch [4/30]
Train Loss: 0.1707, Train Acc: 93.91%
Val Loss: 0.1071, Val Acc: 96.88%
------------------------------
Epoch [5/30]
Train Loss: 0.1152, Train Acc: 95.96%
Val Loss: 0.0770, Val Acc: 97.86%
------------------------------
Epoch [6/30]
Train Loss: 0.0846, Train Acc: 97.17%
Val Loss: 0.0529, Val Acc: 98.48%
------------------------------
Epoch [7/30]
Train Loss: 0.0646, Train Acc: 97.81%
Val Loss: 0.0451, Val Acc: 98.66%
------------------------------
Epoch [8/30]
Train Loss: 0.0638, Train Acc: 97.79%
Val Loss: 0.0406, Val Acc: 98.93%
------------------------------
Epoch [9/30]
Train Loss: 0.0456, Train Acc: 98.62%
Val

In [23]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

model.load_state_dict(torch.load('best_attention_fusion.pth'))
model.eval()

all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')
auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

print(f'Test Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%')
print(f'Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {auc_roc:.4f}')
print(classification_report(all_labels, all_preds, target_names=['glioma', 'meningioma', 'notumor', 'pituitary']))

Test Accuracy: 95.06%
Precision: 0.9540 | Recall: 0.9506 | F1: 0.9498 | AUC: 0.9864
              precision    recall  f1-score   support

      glioma       1.00      0.83      0.91       400
  meningioma       0.89      0.98      0.93       400
     notumor       0.95      1.00      0.97       400
   pituitary       0.98      0.99      0.99       400

    accuracy                           0.95      1600
   macro avg       0.95      0.95      0.95      1600
weighted avg       0.95      0.95      0.95      1600



In [24]:
# This is the code for saving the model into 

import os

# Assume 'model' is your trained PyTorch CNN model
# and 'EPOCH' is the current epoch number.

# Define the save directory and filename
save_dir = r"G:\Reasrch_Paper_2026" # Use 'r' for raw string to handle backslashes
file_name = f"Attention_fusion_model1.pth"
save_path = os.path.join(save_dir, file_name)

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

# Save only the model's state_dict
torch.save(model.state_dict(), save_path)